# Nottingham Forest defender data import

This notebook loads the raw 2025/26 Premier League JSON without modifying it. It creates pandas tables for Forest fixtures, squad members, match events, defender events, and starting-lineup slots.

At this stage, `Defender` is the provider's broad squad label. We are deliberately **not** assigning centre-back/full-back labels yet.

In [ ]:
from pathlib import Path
import json

import matplotlib.patheffects as path_effects
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mplsoccer import VerticalPitch

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

# This works whether the notebook is launched from the project root or notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
DATA_DIRECTORY_NAMES = (
    "raw eventing data for FMP",
    "eventing data",
    "Eventing data for FMP",
)
if not any((PROJECT_ROOT / name).exists() for name in DATA_DIRECTORY_NAMES):
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = next(
    PROJECT_ROOT / name
    for name in DATA_DIRECTORY_NAMES
    if (PROJECT_ROOT / name).exists()
)
SEASON_ROOT = DATA_ROOT / "europa" / "England_Premier_League" / "2025-2026"
FOREST_ROOT = SEASON_ROOT / "equipos" / "Nottingham_Forest_FC"
MATCH_DIR = SEASON_ROOT / "partidos"

assert SEASON_ROOT.exists(), f"Season directory not found: {SEASON_ROOT}"
assert FOREST_ROOT.exists(), f"Forest directory not found: {FOREST_ROOT}"

FOREST_TEAM_ID = "1qtaiy11gswx327s0vkibf70n"
SEASON_ROOT

## Load fixture index and Forest squad

The squad JSON is preferable for names because some names are blank in the derived player CSV.

In [ ]:
matches_index = pd.read_csv(SEASON_ROOT / "matches_ids.csv")

forest_fixtures = matches_index.loc[
    matches_index["equipo_local"].str.contains("Forest", case=False, na=False)
    | matches_index["equipo_visitante"].str.contains("Forest", case=False, na=False)
].copy()

with (FOREST_ROOT / "jsons" / "squad.json").open(encoding="utf-8") as file:
    squad_payload = json.load(file)

squad_people = squad_payload["squad"][0]["person"]
forest_squad = pd.json_normalize(squad_people)
forest_squad = forest_squad.loc[forest_squad["type"].eq("player")].copy()

forest_defenders = forest_squad.loc[
    forest_squad["position"].eq("Defender")
].copy()

print(f"Forest fixtures in index: {len(forest_fixtures)}")
print(f"Players in squad JSON: {len(forest_squad)}")
print(f"Players broadly labelled Defender: {len(forest_defenders)}")

forest_defenders[["id", "matchName", "knownName", "shirtNumber", "active"]].sort_values(
    ["active", "shirtNumber"], ascending=[False, True]
)

## Locate and import Forest match payloads

The fixture index contains 38 Forest matches. The local event directory currently contains 36 matching JSON payloads. The coverage table below makes the missing matches explicit.

In [ ]:
match_file_by_id = {
    path.stem.rsplit("_", 1)[-1]: path
    for path in MATCH_DIR.glob("*.json")
}

forest_fixtures["event_file"] = forest_fixtures["id"].map(match_file_by_id)
forest_fixtures["event_payload_available"] = forest_fixtures["event_file"].notna()

forest_match_payloads = []
for path in forest_fixtures.loc[forest_fixtures["event_payload_available"], "event_file"]:
    with path.open(encoding="utf-8") as file:
        forest_match_payloads.append(json.load(file))

print(f"Available Forest event payloads: {len(forest_match_payloads)} / {len(forest_fixtures)}")
forest_fixtures.loc[
    ~forest_fixtures["event_payload_available"],
    ["matchday", "id", "equipo_local", "equipo_visitante", "date"],
]

## Flatten match events into pandas

Nested qualifiers remain available both in their original list and as a lookup dictionary. We retain provider IDs so every derived record can be traced back to its raw match payload.

In [ ]:
def qualifiers_to_dict(qualifiers):
    return {
        int(item["qualifierId"]): item.get("value", True)
        for item in (qualifiers or [])
    }


event_rows = []
for match in forest_match_payloads:
    match_info = match["matchInfo"]
    contestants = {item["id"]: item for item in match_info["contestant"]}

    for event in match["liveData"].get("event", []):
        contestant = contestants.get(event.get("contestantId"), {})
        event_rows.append(
            {
                "match_id": match_info["id"],
                "match_date": match_info.get("localDate", match_info.get("date")),
                "match_description": match_info.get("description"),
                "event_id": event.get("eventId"),
                "provider_event_id": event.get("id"),
                "type_id": event.get("typeId"),
                "period_id": event.get("periodId"),
                "time_min": event.get("timeMin"),
                "time_sec": event.get("timeSec"),
                "timestamp": event.get("timeStamp"),
                "contestant_id": event.get("contestantId"),
                "contestant_name": contestant.get("name"),
                "home_away": contestant.get("position"),
                "player_id": event.get("playerId"),
                "player_name": event.get("playerName"),
                "outcome": event.get("outcome"),
                "x": event.get("x"),
                "y": event.get("y"),
                "qualifiers": event.get("qualifier", []),
                "qualifier_map": qualifiers_to_dict(event.get("qualifier")),
            }
        )

events = pd.DataFrame(event_rows)
events["match_date"] = pd.to_datetime(events["match_date"], errors="coerce")

forest_events = events.loc[events["contestant_id"].eq(FOREST_TEAM_ID)].copy()
forest_player_events = forest_events.loc[forest_events["player_id"].notna()].copy()

defender_ids = set(forest_defenders["id"])
forest_defender_events = forest_player_events.loc[
    forest_player_events["player_id"].isin(defender_ids)
].copy()

print(f"All events in 36 Forest matches: {len(events):,}")
print(f"Forest events: {len(forest_events):,}")
print(f"Forest player-attributed events: {len(forest_player_events):,}")
print(f"Forest defender-attributed events: {len(forest_defender_events):,}")

forest_defender_events.head()

## Extract initial lineup slots

Provider event type `34` is the initial team setup. Its qualifiers hold the player list, formation code, broad position codes, shirt numbers, and lineup slots. These raw codes are extracted for later validation; this notebook does not yet assume which slots are centre-backs.

In [ ]:
def split_qualifier_values(value):
    if value is None or value is True:
        return []
    return [item.strip() for item in str(value).split(",")]


squad_name_by_id = (
    forest_squad.assign(
        display_name=forest_squad["knownName"].fillna(forest_squad["matchName"])
    )
    .set_index("id")["display_name"]
    .to_dict()
)

lineup_rows = []
forest_team_setups = forest_events.loc[forest_events["type_id"].eq(34)]

for setup in forest_team_setups.itertuples(index=False):
    q = setup.qualifier_map
    player_ids = split_qualifier_values(q.get(30))
    broad_position_codes = split_qualifier_values(q.get(44))
    shirt_numbers = split_qualifier_values(q.get(59))
    lineup_slots = split_qualifier_values(q.get(131))

    for index, player_id in enumerate(player_ids):
        lineup_slot = lineup_slots[index] if index < len(lineup_slots) else None
        lineup_rows.append(
            {
                "match_id": setup.match_id,
                "match_date": setup.match_date,
                "match_description": setup.match_description,
                "formation_code": q.get(130),
                "player_id": player_id,
                "player_name": squad_name_by_id.get(player_id),
                "broad_position_code": (
                    broad_position_codes[index]
                    if index < len(broad_position_codes)
                    else None
                ),
                "shirt_number": (
                    shirt_numbers[index] if index < len(shirt_numbers) else None
                ),
                "lineup_slot": lineup_slot,
                "is_starter": lineup_slot not in {None, "0"},
                "squad_position": forest_squad.set_index("id")["position"].get(player_id),
            }
        )

forest_lineup_slots = pd.DataFrame(lineup_rows)
forest_starting_defenders = forest_lineup_slots.loc[
    forest_lineup_slots["is_starter"]
    & forest_lineup_slots["player_id"].isin(defender_ids)
].copy()

print(f"Initial Forest team setups found: {len(forest_team_setups)}")
print(f"Starting-defender match records: {len(forest_starting_defenders)}")

forest_starting_defenders[[
    "match_date",
    "match_description",
    "formation_code",
    "lineup_slot",
    "player_name",
    "shirt_number",
]].head(20)

## Clean and enrich defender events

The raw event feed mixes football actions with administrative records. This section labels the event types observed for Forest defenders, converts coordinates and pass destinations to numeric values, and creates explicit quality flags.

Raw rows remain available in `forest_defender_events`. The cleaned spatial subset is stored separately as `clean_defender_events`.

In [ ]:
# Perform/Opta event labels used by the defender rows in this dataset.
EVENT_TYPE_NAMES = {
    1: "Pass",
    2: "Offside Pass",
    3: "Take On",
    4: "Foul",
    5: "Out",
    6: "Corner Awarded",
    7: "Tackle",
    8: "Interception",
    10: "Save",
    12: "Clearance",
    13: "Miss",
    14: "Post",
    15: "Attempt Saved",
    16: "Goal",
    17: "Card",
    18: "Player Off",
    19: "Player On",
    43: "Deleted Event",
    44: "Aerial",
    45: "Challenge",
    49: "Ball Recovery",
    50: "Dispossessed",
    51: "Error",
    55: "Offside Provoked",
    56: "Shield Ball",
    61: "Ball Touch",
    74: "Blocked Pass",
    83: "Attempted Tackle",
}

# Analytical taxonomy. Every observed defender event must belong to one family.
EVENT_FAMILY = {
    # Direct actions
    "Pass": "direct_action",
    "Offside Pass": "direct_action",
    "Take On": "direct_action",
    "Foul": "direct_action",
    "Tackle": "direct_action",
    "Attempted Tackle": "direct_action",
    "Interception": "direct_action",
    "Clearance": "direct_action",
    "Aerial": "direct_action",
    "Challenge": "direct_action",
    "Ball Recovery": "direct_action",
    "Ball Touch": "direct_action",
    "Dispossessed": "direct_action",
    "Shield Ball": "direct_action",
    "Blocked Pass": "direct_action",
    "Save": "direct_action",

    # Shot attempts are encoded by their outcomes.
    "Miss": "attacking_outcome",
    "Post": "attacking_outcome",
    "Attempt Saved": "attacking_outcome",
    "Goal": "attacking_outcome",

    # Provider-attributed outcomes rather than direct actions.
    "Corner Awarded": "contextual_outcome",
    "Error": "contextual_outcome",
    "Offside Provoked": "contextual_outcome",

    # These records should never enter an action-location map.
    "Out": "administrative",
    "Card": "administrative",
    "Player Off": "administrative",
    "Player On": "administrative",
    "Deleted Event": "administrative",
}

def qualifier_number(qualifier_map, qualifier_id):
    value = qualifier_map.get(qualifier_id)
    if value is None or value is True:
        return pd.NA
    return pd.to_numeric(value, errors="coerce")


defender_events_enriched = forest_defender_events.copy()

for column in ["type_id", "period_id", "time_min", "time_sec", "x", "y", "outcome"]:
    defender_events_enriched[column] = pd.to_numeric(
        defender_events_enriched[column], errors="coerce"
    )

defender_events_enriched["event_type"] = (
    defender_events_enriched["type_id"].map(EVENT_TYPE_NAMES).fillna("Unknown")
)
defender_events_enriched["event_family"] = (
    defender_events_enriched["event_type"].map(EVENT_FAMILY).fillna("unclassified")
)
defender_events_enriched["match_minute"] = (
    defender_events_enriched["time_min"]
    + defender_events_enriched["time_sec"].fillna(0) / 60
)

# Qualifier 140/141 are the end coordinates used by passes and some other actions.
defender_events_enriched["end_x"] = defender_events_enriched["qualifier_map"].map(
    lambda q: qualifier_number(q, 140)
)
defender_events_enriched["end_y"] = defender_events_enriched["qualifier_map"].map(
    lambda q: qualifier_number(q, 141)
)
defender_events_enriched["pass_length"] = defender_events_enriched["qualifier_map"].map(
    lambda q: qualifier_number(q, 212)
)
defender_events_enriched["pass_angle"] = defender_events_enriched["qualifier_map"].map(
    lambda q: qualifier_number(q, 213)
)

defender_events_enriched["has_valid_start_location"] = (
    defender_events_enriched["x"].between(0, 100)
    & defender_events_enriched["y"].between(0, 100)
)
defender_events_enriched["is_non_spatial_record"] = (
    defender_events_enriched["event_family"].eq("administrative")
)
defender_events_enriched["is_spatial_action"] = (
    defender_events_enriched["player_id"].notna()
    & defender_events_enriched["has_valid_start_location"]
    & ~defender_events_enriched["is_non_spatial_record"]
)

defender_events_enriched[[
    "match_id", "player_name", "event_type", "event_family", "match_minute",
    "x", "y", "end_x", "end_y", "outcome", "is_spatial_action"
]].head()

In [ ]:
valid_spatial_events = defender_events_enriched.loc[
    defender_events_enriched["is_spatial_action"]
].copy()

direct_actions = valid_spatial_events.loc[
    valid_spatial_events["event_family"].eq("direct_action")
].copy()
attacking_outcomes = valid_spatial_events.loc[
    valid_spatial_events["event_family"].eq("attacking_outcome")
].copy()
contextual_outcomes = valid_spatial_events.loc[
    valid_spatial_events["event_family"].eq("contextual_outcome")
].copy()
administrative_events = defender_events_enriched.loc[
    defender_events_enriched["event_family"].eq("administrative")
].copy()

# Primary input for the positional fingerprint: direct involvement only.
positional_actions = direct_actions.copy()

# Backward-compatible name used in earlier exploratory cells.
clean_defender_events = valid_spatial_events

# Keep action subsets separate so high-volume passes do not overwhelm other behaviour.
PASS_TYPE_IDS = {1, 2, 74}
DEFENSIVE_TYPE_IDS = {7, 8, 10, 12, 44, 45, 49, 56, 83}
SHOT_TYPE_IDS = {13, 14, 15, 16}

defender_passes = positional_actions.loc[
    positional_actions["type_id"].isin(PASS_TYPE_IDS)
].copy()
defender_defensive_actions = positional_actions.loc[
    positional_actions["type_id"].isin(DEFENSIVE_TYPE_IDS)
].copy()
defender_shots = attacking_outcomes.loc[
    attacking_outcomes["type_id"].isin(SHOT_TYPE_IDS)
].copy()
take_on_events = positional_actions.loc[
    positional_actions["event_type"].eq("Take On")
].copy()

print(f"Raw defender event rows: {len(forest_defender_events):,}")
print(f"Valid spatial defender events: {len(valid_spatial_events):,}")
print(f"Direct positional actions: {len(positional_actions):,}")
print(f"Attacking outcomes: {len(attacking_outcomes):,}")
print(f"Contextual outcomes: {len(contextual_outcomes):,}")
print(f"Administrative records: {len(administrative_events):,}")
print(f"Pass actions: {len(defender_passes):,}")
print(f"Defensive actions: {len(defender_defensive_actions):,}")
print(f"Shots: {len(defender_shots):,}")

pd.crosstab(
    valid_spatial_events["event_family"],
    valid_spatial_events["event_type"],
).T.sort_index()

## Reconstruct player-match minutes

Starters begin at minute 0. Substitute-on/off events define the other boundaries, and a direct red or second yellow ends an appearance. For now, `minutes_played` uses the provider's elapsed match clock, including stoppage time; it is therefore suitable for sample-quality filters but should be documented before creating conventional per-90 rates.

In [ ]:
match_metadata_rows = []
for match in forest_match_payloads:
    details = match["liveData"]["matchDetails"]
    match_metadata_rows.append(
        {
            "match_id": match["matchInfo"]["id"],
            "match_end_minute": (
                details.get("matchLengthMin", 0)
                + details.get("matchLengthSec", 0) / 60
            ),
        }
    )
match_metadata = pd.DataFrame(match_metadata_rows)
match_end_by_id = match_metadata.set_index("match_id")["match_end_minute"].to_dict()

forest_player_timeline = forest_player_events.copy()
forest_player_timeline["match_minute"] = (
    pd.to_numeric(forest_player_timeline["time_min"], errors="coerce")
    + pd.to_numeric(forest_player_timeline["time_sec"], errors="coerce").fillna(0) / 60
)

sub_on = (
    forest_player_timeline.loc[forest_player_timeline["type_id"].eq(19)]
    .groupby(["match_id", "player_id"])["match_minute"]
    .min()
)
sub_off = (
    forest_player_timeline.loc[forest_player_timeline["type_id"].eq(18)]
    .groupby(["match_id", "player_id"])["match_minute"]
    .min()
)

# Opta qualifier 32 denotes a second yellow and 33 a direct red.
dismissals = forest_player_timeline.loc[
    forest_player_timeline["type_id"].eq(17)
    & forest_player_timeline["qualifier_map"].map(lambda q: 32 in q or 33 in q)
]
dismissal_minute = dismissals.groupby(["match_id", "player_id"])["match_minute"].min()

lineup_appearances = forest_lineup_slots.copy()
lineup_appearances["sub_on_minute"] = [
    sub_on.get((match_id, player_id), pd.NA)
    for match_id, player_id in zip(
        lineup_appearances["match_id"], lineup_appearances["player_id"]
    )
]

# An appearance is a starter or a player for whom a Player On event exists.
player_match_minutes = lineup_appearances.loc[
    lineup_appearances["is_starter"] | lineup_appearances["sub_on_minute"].notna()
].copy()

minute_rows = []
for row in player_match_minutes.itertuples(index=False):
    key = (row.match_id, row.player_id)
    minute_on = 0.0 if row.is_starter else float(sub_on[key])
    possible_off_minutes = [match_end_by_id[row.match_id]]
    if key in sub_off.index:
        possible_off_minutes.append(float(sub_off[key]))
    if key in dismissal_minute.index:
        possible_off_minutes.append(float(dismissal_minute[key]))
    minute_off = min(possible_off_minutes)

    minute_rows.append(
        {
            "match_id": row.match_id,
            "match_date": row.match_date,
            "match_description": row.match_description,
            "formation_code": row.formation_code,
            "player_id": row.player_id,
            "player_name": row.player_name,
            "squad_position": row.squad_position,
            "lineup_slot": row.lineup_slot,
            "is_starter": row.is_starter,
            "minute_on": minute_on,
            "minute_off": minute_off,
            "minutes_played": max(0.0, minute_off - minute_on),
            "was_dismissed": key in dismissal_minute.index,
        }
    )

player_match_minutes = pd.DataFrame(minute_rows)
defender_match_minutes = player_match_minutes.loc[
    player_match_minutes["player_id"].isin(defender_ids)
].copy()

defender_match_minutes[[
    "match_date", "player_name", "is_starter", "minute_on",
    "minute_off", "minutes_played", "was_dismissed"
]].head(20)

## Define eligible positional-map appearances

The prototype prioritizes stable player-match samples for constructing a positional baseline. An appearance is eligible when the defender played at least 60 minutes, approximately two-thirds of regulation time. There is no minimum-action exclusion or low-sample warning at this stage. `action_count` is retained and displayed as transparent sample context; a warning threshold can be calibrated from the league-wide distribution later.

In [ ]:
MIN_ELIGIBLE_MINUTES = 60

# Count only the direct actions selected for the positional fingerprint.
defender_match_action_counts = (
    positional_actions.groupby(["match_id", "player_id"])
    .size()
    .rename("action_count")
    .reset_index()
)

defender_appearance_samples = defender_match_minutes.merge(
    defender_match_action_counts,
    on=["match_id", "player_id"],
    how="left",
    validate="one_to_one",
)
defender_appearance_samples["action_count"] = (
    defender_appearance_samples["action_count"].fillna(0).astype(int)
)
defender_appearance_samples["is_eligible_appearance"] = (
    defender_appearance_samples["minutes_played"] >= MIN_ELIGIBLE_MINUTES
)

eligible_defender_appearances = defender_appearance_samples.loc[
    defender_appearance_samples["is_eligible_appearance"]
].copy()

# Attach the sample context needed beside each eventual positional map.
eligible_positional_actions = positional_actions.merge(
    eligible_defender_appearances[[
        "match_id", "player_id", "minutes_played", "action_count"
    ]],
    on=["match_id", "player_id"],
    how="inner",
    validate="many_to_one",
)

print(f"Eligible defender appearances: {len(eligible_defender_appearances):,}")
print(f"Eligible positional actions: {len(eligible_positional_actions):,}")

eligible_defender_appearances[[
    "match_date", "match_description", "player_name",
    "minutes_played", "action_count", "is_eligible_appearance"
]].sort_values(["match_date", "player_name"]).head(20)

## Data-quality checks

These checks fail loudly for identifiers that must be unique and summarize other issues that require an analytical decision rather than silent deletion.

In [ ]:
provider_id_duplicates = defender_events_enriched["provider_event_id"].duplicated().sum()
team_sequence_duplicates = defender_events_enriched.duplicated(
    ["match_id", "contestant_id", "event_id"]
).sum()

name_conflicts = (
    defender_events_enriched.dropna(subset=["player_id", "player_name"])
    .groupby("player_id")["player_name"]
    .nunique()
    .gt(1)
    .sum()
)

quality_checks = pd.DataFrame(
    [
        ("Available Forest match payloads", len(forest_match_payloads)),
        ("Missing Forest match payloads", (~forest_fixtures["event_payload_available"]).sum()),
        ("Raw defender rows", len(defender_events_enriched)),
        ("Valid spatial defender events", len(valid_spatial_events)),
        ("Direct positional actions", len(positional_actions)),
        ("Attacking outcomes", len(attacking_outcomes)),
        ("Contextual outcomes", len(contextual_outcomes)),
        ("Administrative records", len(administrative_events)),
        ("Out-of-pitch coordinate rows", (~defender_events_enriched["has_valid_start_location"]).sum()),
        ("Unknown defender event types", defender_events_enriched["event_type"].eq("Unknown").sum()),
        ("Unclassified defender event families", defender_events_enriched["event_family"].eq("unclassified").sum()),
        ("Duplicate provider event IDs", provider_id_duplicates),
        ("Duplicate match/team event IDs", team_sequence_duplicates),
        ("Player IDs with conflicting names", name_conflicts),
        ("Matches missing initial Forest setup", len(forest_match_payloads) - forest_team_setups["match_id"].nunique()),
        ("Defender appearances reconstructed", len(defender_match_minutes)),
        ("Eligible defender appearances (60+ minutes)", len(eligible_defender_appearances)),
        ("Eligible positional actions", len(eligible_positional_actions)),
        ("Non-positive defender minutes", defender_match_minutes["minutes_played"].le(0).sum()),
    ],
    columns=["check", "value"],
)

assert provider_id_duplicates == 0, "provider_event_id must be unique"
assert team_sequence_duplicates == 0, "match/team/event_id must be unique"
assert defender_events_enriched["event_type"].ne("Unknown").all(), "Map all observed event types"
assert defender_events_enriched["event_family"].ne("unclassified").all(), "Classify all observed event types"
assert eligible_defender_appearances["minutes_played"].ge(MIN_ELIGIBLE_MINUTES).all()
assert eligible_defender_appearances["action_count"].ge(0).all()

quality_checks

## Clean analytical tables

The notebook currently provides the following pandas DataFrames.

### Competition, fixtures and squad

- `matches_index`: the complete 2025/26 Premier League fixture index.
- `forest_fixtures`: Forest's 38 fixtures with local event-payload availability.
- `forest_squad`: Forest's provider squad records, normalized from `squad.json`.
- `forest_defenders`: squad members carrying the provider's broad `Defender` label.

### Raw and progressively filtered events

- `events`: both teams' events from the 36 available Forest match payloads.
- `forest_events`: all events attributed to Nottingham Forest.
- `forest_player_events`: Forest events containing a player ID.
- `forest_defender_events`: player-attributed events belonging to broadly labelled Forest defenders; this is the raw defender-event source.
- `defender_events_enriched`: defender events with readable event types, event families, numeric time/coordinates, pass destinations and quality flags.

### Event-taxonomy views

- `valid_spatial_events`: all defender football events with valid on-pitch start coordinates, excluding administrative records.
- `direct_actions`: passes, take-ons, tackles, interceptions, clearances, aerials, recoveries, touches and other direct actions.
- `attacking_outcomes`: defender shots recorded as misses, posts, saved attempts or goals.
- `contextual_outcomes`: corners awarded, provider-labelled errors and offsides provoked.
- `administrative_events`: cards, substitutions, ball-out records and deleted events.
- `positional_actions`: the direct-action input selected for the primary positional fingerprint.
- `clean_defender_events`: backward-compatible alias for `valid_spatial_events`; use the more explicit name in new analysis.

### Action-specific views

- `defender_passes`: passes, offside passes and blocked passes from `positional_actions`.
- `defender_defensive_actions`: tackles, attempted tackles, interceptions, clearances, aerials, challenges, recoveries, saves and shields.
- `defender_shots`: all four attacking-outcome types.
- `take_on_events`: all defender take-on attempts.
- `successful_take_ons`: successful take-on attempts created in the exploratory section.

### Lineups, formations and playing time

- `forest_team_setups`: Forest's initial team-setup events (`type_id == 34`).
- `forest_lineup_slots`: one record per listed player and match, retaining raw formation and lineup-slot codes.
- `forest_starting_defenders`: broadly labelled defenders who started each match.
- `match_metadata`: match IDs and provider-reported elapsed match lengths.
- `forest_player_timeline`: Forest player events with a calculated elapsed `match_minute`.
- `dismissals`: second-yellow and direct-red events used when calculating playing time.
- `lineup_appearances`: lineup records augmented with substitute-on information.
- `player_match_minutes`: reconstructed minutes for every Forest appearance.
- `defender_match_minutes`: reconstructed minutes for broadly labelled defenders.
- `defender_match_action_counts`: direct positional-action counts for each defender and match.
- `defender_appearance_samples`: defender minutes joined to `action_count` and the 60-minute eligibility flag.
- `eligible_defender_appearances`: appearances satisfying `minutes_played >= 60`; no action-count exclusion is applied.
- `eligible_positional_actions`: direct actions from eligible appearances, with minutes and action counts attached for map labels.

### Validation

- `quality_checks`: reproducible coverage, classification, identifier, lineup and playing-time checks.

### First positional-map prototype

- `shot_orientation_check`: shot-location evidence that both home and away teams attack toward increasing Opta `x` in both halves.
- `selected_appearance`: metadata for the prototype player-match sample.
- `selected_appearance_actions`: its direct action locations.
- `positional_20_distribution`: the normalized 20-zone positional distribution.
- `traditional_18_distribution`: the normalized equal-sized 6-by-3 distribution.

The two prototype maps show shares of recorded direct-action locations, not continuous time spent in each zone.

In [ ]:
successful_take_ons = take_on_events.loc[
    take_on_events["outcome"].eq(1)
].copy()

successful_take_ons[[
    "match_date", "match_description", "player_name",
    "match_minute", "x", "y", "outcome", "provider_event_id"
]].sort_values(["match_date", "match_minute"])

## Validate the Opta attacking direction

Zone numbers only have a consistent football meaning if `x = 0` represents a team's own goal and `x = 100` the opponent's goal. The check below uses all shots in the available Forest matches. If home and away shots in both halves remain concentrated above `x = 50`, the feed is normalized to each team's attacking direction and no half-by-half mirroring is required.

In [ ]:
ALL_SHOT_TYPE_IDS = {13, 14, 15, 16}

orientation_shots = events.loc[
    pd.to_numeric(events["type_id"], errors="coerce").isin(ALL_SHOT_TYPE_IDS)
    & pd.to_numeric(events["period_id"], errors="coerce").isin([1, 2])
].copy()
orientation_shots["x"] = pd.to_numeric(orientation_shots["x"], errors="coerce")
orientation_shots["period_id"] = pd.to_numeric(
    orientation_shots["period_id"], errors="coerce"
).astype("Int64")
orientation_shots = orientation_shots.loc[orientation_shots["x"].between(0, 100)]

shot_orientation_check = (
    orientation_shots.groupby(["home_away", "period_id"], observed=True)
    .agg(
        shot_count=("x", "size"),
        median_x=("x", "median"),
        mean_x=("x", "mean"),
        minimum_x=("x", "min"),
        maximum_x=("x", "max"),
    )
    .reset_index()
)
shot_orientation_check["share_above_halfway"] = [
    group["x"].gt(50).mean()
    for _, group in orientation_shots.groupby(
        ["home_away", "period_id"], observed=True, sort=True
    )
]

assert shot_orientation_check["median_x"].gt(50).all(), (
    "Shot locations do not consistently point toward x=100; investigate mirroring."
)

shot_orientation_check

## First positional-map prototype: Murillo vs Brentford

This prototype compares two divisions of the same Opta action locations: the tactical 20-zone positional layout and a traditional 18-zone grid made from six equal length bands and three equal width bands. Every value is the share of Murillo's recorded direct actions beginning in that zone. It is not an estimate of continuous physical occupancy.

In [ ]:
SELECTED_MATCH_ID = "110stnyllsj1u0mhdabeuvyfo"
SELECTED_PLAYER_ID = "4txq9os03xwnktfi0n2qdmok"

selected_rows = eligible_defender_appearances.loc[
    eligible_defender_appearances["match_id"].eq(SELECTED_MATCH_ID)
    & eligible_defender_appearances["player_id"].eq(SELECTED_PLAYER_ID)
]
assert len(selected_rows) == 1, "Expected one eligible Murillo appearance"
selected_appearance = selected_rows.iloc[0].copy()

selected_appearance_actions = eligible_positional_actions.loc[
    eligible_positional_actions["match_id"].eq(SELECTED_MATCH_ID)
    & eligible_positional_actions["player_id"].eq(SELECTED_PLAYER_ID)
].copy()

assert len(selected_appearance_actions) == int(selected_appearance["action_count"])
assert selected_appearance_actions["x"].between(0, 100).all()
assert selected_appearance_actions["y"].between(0, 100).all()

pitch = VerticalPitch(
    pitch_type="opta",
    pitch_color="#22312b",
    line_color="white",
    line_zorder=2,
)

positional_20_bins = pitch.bin_statistic_positional(
    selected_appearance_actions["x"],
    selected_appearance_actions["y"],
    statistic="count",
    positional="full",
    normalize=True,
)
positional_20_distribution = pd.DataFrame(
    {
        "zone_id": np.arange(1, 21),
        "zone_name": positional_20_bins["names"],
        "center_x": positional_20_bins["cx"],
        "center_y": positional_20_bins["cy"],
        "action_count": positional_20_bins["count"].astype(int),
        "action_share": positional_20_bins["statistic"],
    }
)

traditional_18_bins = pitch.bin_statistic(
    selected_appearance_actions["x"],
    selected_appearance_actions["y"],
    statistic="count",
    bins=(6, 3),
    normalize=True,
)

traditional_actions = selected_appearance_actions[["x", "y"]].copy()
traditional_actions["length_band"] = np.minimum(
    np.floor(traditional_actions["x"] / (100 / 6)).astype(int), 5
)
traditional_actions["width_band"] = np.minimum(
    np.floor(traditional_actions["y"] / (100 / 3)).astype(int), 2
)
traditional_actions["zone_id"] = (
    traditional_actions["length_band"] * 3
    + (2 - traditional_actions["width_band"])
    + 1
)
traditional_18_distribution = (
    traditional_actions.groupby("zone_id")
    .size()
    .reindex(range(1, 19), fill_value=0)
    .rename("action_count")
    .reset_index()
)
traditional_18_distribution["action_share"] = (
    traditional_18_distribution["action_count"]
    / traditional_18_distribution["action_count"].sum()
)
traditional_18_distribution["length_band"] = (
    (traditional_18_distribution["zone_id"] - 1) // 3
)
traditional_18_distribution["width_band"] = (
    (traditional_18_distribution["zone_id"] - 1) % 3
)
traditional_18_distribution["center_x"] = (
    traditional_18_distribution["length_band"] + 0.5
) * (100 / 6)
traditional_18_distribution["center_y"] = (
    2.5 - traditional_18_distribution["width_band"]
) * (100 / 3)

assert positional_20_distribution["action_count"].sum() == len(selected_appearance_actions)
assert traditional_18_distribution["action_count"].sum() == len(selected_appearance_actions)
assert np.isclose(positional_20_distribution["action_share"].sum(), 1)
assert np.isclose(traditional_18_distribution["action_share"].sum(), 1)

shared_vmax = max(
    np.nanmax(positional_20_bins["statistic"]),
    np.nanmax(traditional_18_bins["statistic"]),
)
label_effects = [
    path_effects.Stroke(linewidth=2.5, foreground="black"),
    path_effects.Normal(),
]

fig, axes = pitch.draw(nrows=1, ncols=2, figsize=(12, 8))
heatmap_20 = pitch.heatmap_positional(
    positional_20_bins,
    ax=axes[0],
    cmap="cividis",
    edgecolors="#22312b",
    vmin=0,
    vmax=shared_vmax,
)
pitch.scatter(
    selected_appearance_actions["x"],
    selected_appearance_actions["y"],
    ax=axes[0],
    color="white",
    s=8,
    alpha=0.35,
    zorder=3,
)
for zone in positional_20_distribution.itertuples(index=False):
    pitch.annotate(
        f"Z{zone.zone_id:02d}\n{zone.action_share:.1%}",
        xy=(zone.center_x, zone.center_y),
        ax=axes[0],
        color="white",
        fontsize=8.5,
        fontweight="bold",
        ha="center",
        va="center",
        path_effects=label_effects,
        zorder=4,
    )
axes[0].set_title("Positional 20-zone layout", color="white", fontsize=14, pad=10)

pitch.heatmap(
    traditional_18_bins,
    ax=axes[1],
    cmap="cividis",
    edgecolors="#22312b",
    vmin=0,
    vmax=shared_vmax,
)
pitch.scatter(
    selected_appearance_actions["x"],
    selected_appearance_actions["y"],
    ax=axes[1],
    color="white",
    s=8,
    alpha=0.35,
    zorder=3,
)
for zone in traditional_18_distribution.itertuples(index=False):
    pitch.annotate(
        f"Z{zone.zone_id:02d}\n{zone.action_share:.1%}",
        xy=(zone.center_x, zone.center_y),
        ax=axes[1],
        color="white",
        fontsize=8.5,
        fontweight="bold",
        ha="center",
        va="center",
        path_effects=label_effects,
        zorder=4,
    )
axes[1].set_title("Traditional 18-zone layout", color="white", fontsize=14, pad=10)

fig.patch.set_facecolor("#22312b")
fig.suptitle(
    f"{selected_appearance['player_name']} — {selected_appearance['match_description']}",
    color="white",
    fontsize=17,
    y=0.98,
)
fig.text(
    0.5,
    0.94,
    (
        f"{selected_appearance['match_date']:%d %B %Y} | "
        f"{selected_appearance['minutes_played']:.1f} minutes | "
        f"{int(selected_appearance['action_count'])} direct actions"
    ),
    color="#d9e2df",
    fontsize=11,
    ha="center",
)
fig.text(
    0.5, 0.02, "Values show each zone's share of recorded direct-action start locations.",
    color="#d9e2df", fontsize=10, ha="center"
)
fig.subplots_adjust(top=0.88, bottom=0.07, wspace=0.08)
plt.show()

positional_20_distribution, traditional_18_distribution